In [26]:
import pandas as pd
import random

In [27]:
# Load the dataset
df = pd.read_csv('data/soc-sign-bitcoinotc.csv.gz', names=["SOURCE", "TARGET", "RATING", "TIME"])

# Extract nodes and edges
nodes = set(df['SOURCE']).union(set(df['TARGET']))  # Unique nodes
edges = set(tuple(row) for row in df[['SOURCE', 'TARGET']].to_numpy())  # Edges as tuples

print("Number of Nodes:", len(nodes))
print("Number of Edges:", len(edges))


Number of Nodes: 5881
Number of Edges: 35592


In [30]:
def flas_algorithm(nodes, edges, ns, G2N_automaton):
    """
    FLAS Algorithm for graph sampling with corrected loop conditions.
    
    Parameters:
    - nodes: Set of nodes in the original graph.
    - edges: Set of edges in the original graph as tuples (SOURCE, TARGET).
    - ns: Desired sample size.
    - G2N_automaton: A function or class instance representing the automaton logic.
    
    Returns:
    - Sampled subgraph as a tuple (sampled_nodes, sampled_edges).
    """
    # Initialization part
    Vs = set()  # Sampled nodes
    Es = set()  # Sampled edges
    t = 0  # Time index

    edge_list = list(edges)
    num_edges = len(edge_list)
    if num_edges == 0:
        return Vs, Es  # Return empty if no edges are available

    # Step 1: Initialize sample until reaching ns nodes
    while len(Vs) < ns and t < num_edges:
        vt, vl = edge_list[t]
        Es.add((vt, vl))
        Vs.update([vt, vl])  # Add both nodes to the sample

        for node in [vt, vl]:
            if node not in G2N_automaton.states:  # Assign automaton to unseen nodes
                G2N_automaton.assign_automaton(node)

        t += 1

    # Step 2: Update loop (streaming edges)
    while t < num_edges:
        vt, vl = edge_list[t]

        for node in [vt, vl]:
            if node not in Vs:
                # New node: Assign automaton
                G2N_automaton.assign_automaton(node)

                # Penalize or reward logic
                if random.random() <= 0.5:  # Example probability
                    G2N_automaton.reward_automaton(node)
                else:
                    G2N_automaton.penalize_automaton(node)

        # Add edge if both nodes are in the sample
        if vt in Vs and vl in Vs:
            Es.add((vt, vl))
        else:
            Vs.add(vt)
            Vs.add(vl)

        # Check if we exceeded the sample size
        while len(Vs) > ns:
            # Choose a node to remove (e.g., randomly or based on automaton state)
            to_remove = random.choice(list(Vs))
            Vs.remove(to_remove)
            Es = {e for e in Es if to_remove not in e}  # Remove edges connected to this node

        t += 1  # Increment time

        # Break early if we've already reached the desired sample size and no new edges can be processed
        if len(Es) >= num_edges:
            break

    return Vs, Es


# Example usage
class G2N2Automaton:
    def __init__(self):
        self.states = {}

    def assign_automaton(self, node):
        self.states[node] = random.randint(0, 10)  # Example state initialization

    def reward_automaton(self, node):
        self.states[node] = max(0, self.states[node] - 1)

    def penalize_automaton(self, node):
        self.states[node] += 1


# Assuming `nodes` and `edges` are extracted from your dataset
ns = 100  # Desired sample size
automaton = G2N2Automaton()
sampled_nodes, sampled_edges = flas_algorithm(nodes, edges, ns, automaton)

print("Sampled Nodes:", len(sampled_nodes))
print("Sampled Edges:", len(sampled_edges))


Sampled Nodes: 100
Sampled Edges: 0
